# MLP optimization

Compare SGD, Adam and AdamW, then vary schedules, weight decay, dropout, batch normalization, batch size, gradient clipping and early stopping.

In [ ]:
from pathlib import Path
import pandas as pd

from scania_aps.data import TRAIN_FILENAME, TEST_FILENAME, read_raw_csv

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"
train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)
print(train.X.shape, test.X.shape, train.y.mean(), test.y.mean())

In [ ]:
from scania_aps.model_search import candidates_for_family
from scania_aps.models.factory import build_candidate
from scania_aps.split import research_split
from scania_aps.scoring import positive_class_scores
from scania_aps.costs import optimize_score_threshold

split = research_split(train.X, train.y)
rows=[]
for candidate in candidates_for_family("mlp", profile="quick"):
    model=build_candidate(candidate).fit(split.X_fit, split.y_fit)
    scores=positive_class_scores(model, split.X_tune)
    cost=optimize_score_threshold(split.y_tune.to_numpy(), scores.values).cost.total_cost
    history=model.named_steps["model"].history_
    rows.append({"name": candidate.name, "cost": cost, "epochs": len(history), **candidate.parameters})
pd.DataFrame(rows).sort_values("cost")